## 01_pancreas_manifest_y_subsets.ipynb
 Objetivo:
 1) localizar imágenes .nii.gz del batch actual de PANORAMA
 2) cargar clinical_information.xlsx
 3) construir manifest por estudio (case_id)
 4) mapear etiqueta binaria PDAC / non-PDAC
 5) crear split train/val/test por paciente para evitar leakage

## imports

In [18]:
from pathlib import Path
import random
import numpy as np
import pandas as pd
from sklearn.model_selection import train_test_split
from IPython.display import display

## configuración

In [19]:
SEED = 42
random.seed(SEED)
np.random.seed(SEED)

PROJECT_ROOT = Path("/mnt/d/Universidad/analitica/proyecto_analitica2")
DATA_DIR = PROJECT_ROOT / "data"
SOURCE_DIR = DATA_DIR / "source"
WORKING_DIR = DATA_DIR / "working"
MANIFESTS_DIR = DATA_DIR / "manifests"

PANCREAS_DIR = SOURCE_DIR / "pancreas"
PANORAMA_LABELS_DIR = PANCREAS_DIR / "panorama_labels"

PANCREAS_WORKING_DIR = WORKING_DIR / "pancreas"
PANCREAS_WORKING_DIR.mkdir(parents=True, exist_ok=True)
MANIFESTS_DIR.mkdir(parents=True, exist_ok=True)

CLINICAL_PATH = PANORAMA_LABELS_DIR / "clinical_information.xlsx"

print("PANCREAS_DIR:", PANCREAS_DIR)
print("PANORAMA_LABELS_DIR:", PANORAMA_LABELS_DIR)
print("CLINICAL_PATH existe:", CLINICAL_PATH.exists())

PANCREAS_DIR: /mnt/d/Universidad/analitica/proyecto_analitica2/data/source/pancreas
PANORAMA_LABELS_DIR: /mnt/d/Universidad/analitica/proyecto_analitica2/data/source/pancreas/panorama_labels
CLINICAL_PATH existe: True


## localizar imágenes del batch actual

In [20]:
image_files = sorted(PANCREAS_DIR.glob("*.nii.gz"))

image_df = pd.DataFrame({
    "image_name": [p.name for p in image_files],
    "image_path": [str(p.resolve()) for p in image_files],
})

image_df["case_id"] = image_df["image_name"].str.replace("_0000.nii.gz", "", regex=False)
image_df["patient_id"] = image_df["case_id"].str.split("_").str[0]
image_df["study_suffix"] = image_df["case_id"].str.split("_").str[1]

print("Cantidad imágenes encontradas:", len(image_df))
display(image_df.head())

Cantidad imágenes encontradas: 557


,image_name,image_path,case_id,patient_id,study_suffix
0,100000_00001_0000.nii.gz,/mnt/d/Universidad/analitica/proyecto_analitic...,100000_00001,100000,00001
1,100001_00001_0000.nii.gz,/mnt/d/Universidad/analitica/proyecto_analitic...,100001_00001,100001,00001
2,100002_00001_0000.nii.gz,/mnt/d/Universidad/analitica/proyecto_analitic...,100002_00001,100002,00001
3,100003_00001_0000.nii.gz,/mnt/d/Universidad/analitica/proyecto_analitic...,100003_00001,100003,00001
4,100004_00001_0000.nii.gz,/mnt/d/Universidad/analitica/proyecto_analitic...,100004_00001,100004,00001


## cargar clinical_information.xlsx

In [21]:
df_clin = pd.read_excel(CLINICAL_PATH, sheet_name=0)

print("Shape clinical:", df_clin.shape)
print("Columnas:", df_clin.columns.tolist())
display(df_clin.head())

Shape clinical: (2238, 8)
Columnas: ['PANORAMA_patient_id', 'PANORAMA_study_id', 'anonymized_study_date', 'patient_age', 'patient_sex', 'scanner', 'label', 'level']


,PANORAMA_patient_id,PANORAMA_study_id,anonymized_study_date,patient_age,patient_sex,scanner,label,level
0,100000,100000_00001,2018-10-04,042Y,F,TOSHIBA,non-PDAC,radiology
1,100001,100001_00001,2014-04-24,068Y,M,SIEMENS,non-PDAC,radiology
2,100002,100002_00001,2021-03-02,077Y,F,TOSHIBA,PDAC,pathology
3,100003,100003_00001,2016-12-28,057Y,M,TOSHIBA,PDAC,cytology
4,100004,100004_00001,2018-02-09,068Y,M,SIEMENS,non-PDAC,radiology


## preparar tabla clínica para merge

In [22]:
clin_df = df_clin.copy()

clin_df["PANORAMA_patient_id"] = clin_df["PANORAMA_patient_id"].astype(str)
clin_df["PANORAMA_study_id"] = clin_df["PANORAMA_study_id"].astype(str)

clin_df["case_id"] = clin_df["PANORAMA_study_id"]
clin_df["patient_id"] = clin_df["PANORAMA_patient_id"]

clin_df["label_str"] = clin_df["label"].astype(str).str.strip()
clin_df["label_binary"] = clin_df["label_str"].map({
    "non-PDAC": 0,
    "PDAC": 1
})

print("Distribución global de label_str en clinical:")
print(clin_df["label_str"].value_counts(dropna=False))

print("\nValores faltantes en label_binary:", clin_df["label_binary"].isna().sum())
display(clin_df.head())

Distribución global de label_str en clinical:
label_str
non-PDAC    1562
PDAC         676
Name: count, dtype: int64

Valores faltantes en label_binary: 0


,PANORAMA_patient_id,PANORAMA_study_id,anonymized_study_date,patient_age,patient_sex,scanner,label,level,case_id,patient_id,label_str,label_binary
0,100000,100000_00001,2018-10-04,042Y,F,TOSHIBA,non-PDAC,radiology,100000_00001,100000,non-PDAC,0
1,100001,100001_00001,2014-04-24,068Y,M,SIEMENS,non-PDAC,radiology,100001_00001,100001,non-PDAC,0
2,100002,100002_00001,2021-03-02,077Y,F,TOSHIBA,PDAC,pathology,100002_00001,100002,PDAC,1
3,100003,100003_00001,2016-12-28,057Y,M,TOSHIBA,PDAC,cytology,100003_00001,100003,PDAC,1
4,100004,100004_00001,2018-02-09,068Y,M,SIEMENS,non-PDAC,radiology,100004_00001,100004,non-PDAC,0


## merge imágenes ↔ clínica

In [23]:
manifest = image_df.merge(
    clin_df[
        [
            "case_id",
            "patient_id",
            "label_str",
            "label_binary",
            "level",
            "patient_age",
            "patient_sex",
            "scanner",
            "anonymized_study_date",
        ]
    ],
    on=["case_id", "patient_id"],
    how="left"
)

print("Shape manifest después del merge:", manifest.shape)
print("\nCasos sin label clínica:", manifest["label_binary"].isna().sum())
display(manifest.head())

Shape manifest después del merge: (557, 12)

Casos sin label clínica: 0


,image_name,image_path,case_id,patient_id,study_suffix,label_str,label_binary,level,patient_age,patient_sex,scanner,anonymized_study_date
0,100000_00001_0000.nii.gz,/mnt/d/Universidad/analitica/proyecto_analitic...,100000_00001,100000,00001,non-PDAC,0,radiology,042Y,F,TOSHIBA,2018-10-04
1,100001_00001_0000.nii.gz,/mnt/d/Universidad/analitica/proyecto_analitic...,100001_00001,100001,00001,non-PDAC,0,radiology,068Y,M,SIEMENS,2014-04-24
2,100002_00001_0000.nii.gz,/mnt/d/Universidad/analitica/proyecto_analitic...,100002_00001,100002,00001,PDAC,1,pathology,077Y,F,TOSHIBA,2021-03-02
3,100003_00001_0000.nii.gz,/mnt/d/Universidad/analitica/proyecto_analitic...,100003_00001,100003,00001,PDAC,1,cytology,057Y,M,TOSHIBA,2016-12-28
4,100004_00001_0000.nii.gz,/mnt/d/Universidad/analitica/proyecto_analitic...,100004_00001,100004,00001,non-PDAC,0,radiology,068Y,M,SIEMENS,2018-02-09


## agregar paths opcionales de masks manuales y automáticas

In [24]:
manual_dir = PANORAMA_LABELS_DIR / "manual_labels"
auto_dir = PANORAMA_LABELS_DIR / "automatic_labels"

manifest["manual_label_path"] = manifest["case_id"].apply(
    lambda x: str((manual_dir / f"{x}.nii.gz").resolve()) if (manual_dir / f"{x}.nii.gz").exists() else None
)

manifest["automatic_label_path"] = manifest["case_id"].apply(
    lambda x: str((auto_dir / f"{x}.nii.gz").resolve()) if (auto_dir / f"{x}.nii.gz").exists() else None
)

print("Casos con manual_label_path:", manifest["manual_label_path"].notna().sum())
print("Casos con automatic_label_path:", manifest["automatic_label_path"].notna().sum())

Casos con manual_label_path: 108
Casos con automatic_label_path: 449


## filtros de integridad

In [25]:
manifest = manifest.dropna(subset=["label_binary"]).copy()
manifest["label_binary"] = manifest["label_binary"].astype(int)

manifest = manifest.drop_duplicates(subset=["case_id"]).reset_index(drop=True)

print("Manifest final tras filtros:", manifest.shape)
print("\nDistribución final de clases:")
print(manifest["label_binary"].value_counts())

display(
    manifest[
        [
            "case_id",
            "patient_id",
            "image_name",
            "label_str",
            "label_binary",
            "level",
            "manual_label_path",
            "automatic_label_path",
        ]
    ].head()
)

Manifest final tras filtros: (557, 14)

Distribución final de clases:
label_binary
0    399
1    158
Name: count, dtype: int64


,case_id,patient_id,image_name,label_str,label_binary,level,manual_label_path,automatic_label_path
0,100000_00001,100000,100000_00001_0000.nii.gz,non-PDAC,0,radiology,None,/mnt/d/Universidad/analitica/proyecto_analitic...
1,100001_00001,100001,100001_00001_0000.nii.gz,non-PDAC,0,radiology,None,/mnt/d/Universidad/analitica/proyecto_analitic...
2,100002_00001,100002,100002_00001_0000.nii.gz,PDAC,1,pathology,/mnt/d/Universidad/analitica/proyecto_analitic...,None
3,100003_00001,100003,100003_00001_0000.nii.gz,PDAC,1,cytology,None,/mnt/d/Universidad/analitica/proyecto_analitic...
4,100004_00001,100004,100004_00001_0000.nii.gz,non-PDAC,0,radiology,None,/mnt/d/Universidad/analitica/proyecto_analitic...


## chequeo de consistencia por paciente

In [26]:
patient_label_nunique = (
    manifest.groupby("patient_id")["label_binary"]
    .nunique()
    .reset_index(name="n_unique_labels")
)

inconsistent_patients = patient_label_nunique[patient_label_nunique["n_unique_labels"] > 1]

print("Pacientes totales:", manifest["patient_id"].nunique())
print("Pacientes con labels inconsistentes:", len(inconsistent_patients))

if len(inconsistent_patients) > 0:
    display(inconsistent_patients.head(20))

Pacientes totales: 547
Pacientes con labels inconsistentes: 0


## construir tabla de pacientes para split sin leakage

In [27]:
patient_df = (
    manifest.groupby("patient_id")
    .agg(
        label_binary=("label_binary", "first"),
        n_studies=("case_id", "count"),
    )
    .reset_index()
)

print("Shape patient_df:", patient_df.shape)
print("\nDistribución por clase a nivel paciente:")
print(patient_df["label_binary"].value_counts())

display(patient_df.head())

Shape patient_df: (547, 3)

Distribución por clase a nivel paciente:
label_binary
0    391
1    156
Name: count, dtype: int64


,patient_id,label_binary,n_studies
0,100000,0,1
1,100001,0,1
2,100002,1,1
3,100003,1,1
4,100004,0,1


## split por paciente, estratificado

In [28]:
train_patients, temp_patients = train_test_split(
    patient_df,
    test_size=0.30,
    random_state=SEED,
    stratify=patient_df["label_binary"]
)

val_patients, test_patients = train_test_split(
    temp_patients,
    test_size=0.50,
    random_state=SEED,
    stratify=temp_patients["label_binary"]
)

train_patients = train_patients.copy()
val_patients = val_patients.copy()
test_patients = test_patients.copy()

train_patients["split"] = "train"
val_patients["split"] = "val"
test_patients["split"] = "test"

patient_split_df = pd.concat(
    [train_patients, val_patients, test_patients],
    axis=0
).reset_index(drop=True)

print("Conteo de pacientes por split:")
print(patient_split_df["split"].value_counts())

print("\nDistribución paciente por split y clase:")
print(pd.crosstab(patient_split_df["split"], patient_split_df["label_binary"]))

Conteo de pacientes por split:
split
train    382
test      83
val       82
Name: count, dtype: int64

Distribución paciente por split y clase:
label_binary    0    1
split                 
test           59   24
train         273  109
val            59   23


## llevar split de paciente a estudio

In [29]:
split_df = manifest.merge(
    patient_split_df[["patient_id", "split"]],
    on="patient_id",
    how="left"
).copy()

print("Conteo de estudios por split:")
print(split_df["split"].value_counts())

print("\nDistribución de estudios por split y clase:")
print(pd.crosstab(split_df["split"], split_df["label_binary"]))

display(split_df.head())

Conteo de estudios por split:
split
train    392
test      83
val       82
Name: count, dtype: int64

Distribución de estudios por split y clase:
label_binary    0    1
split                 
test           59   24
train         281  111
val            59   23


,image_name,image_path,case_id,patient_id,study_suffix,label_str,label_binary,level,patient_age,patient_sex,scanner,anonymized_study_date,manual_label_path,automatic_label_path,split
0,100000_00001_0000.nii.gz,/mnt/d/Universidad/analitica/proyecto_analitic...,100000_00001,100000,00001,non-PDAC,0,radiology,042Y,F,TOSHIBA,2018-10-04,None,/mnt/d/Universidad/analitica/proyecto_analitic...,train
1,100001_00001_0000.nii.gz,/mnt/d/Universidad/analitica/proyecto_analitic...,100001_00001,100001,00001,non-PDAC,0,radiology,068Y,M,SIEMENS,2014-04-24,None,/mnt/d/Universidad/analitica/proyecto_analitic...,val
2,100002_00001_0000.nii.gz,/mnt/d/Universidad/analitica/proyecto_analitic...,100002_00001,100002,00001,PDAC,1,pathology,077Y,F,TOSHIBA,2021-03-02,/mnt/d/Universidad/analitica/proyecto_analitic...,None,val
3,100003_00001_0000.nii.gz,/mnt/d/Universidad/analitica/proyecto_analitic...,100003_00001,100003,00001,PDAC,1,cytology,057Y,M,TOSHIBA,2016-12-28,None,/mnt/d/Universidad/analitica/proyecto_analitic...,train
4,100004_00001_0000.nii.gz,/mnt/d/Universidad/analitica/proyecto_analitic...,100004_00001,100004,00001,non-PDAC,0,radiology,068Y,M,SIEMENS,2018-02-09,None,/mnt/d/Universidad/analitica/proyecto_analitic...,train


## resumen de estudios múltiples por paciente

In [30]:
multi_study_df = (
    manifest.groupby("patient_id")
    .size()
    .reset_index(name="n_studies")
    .sort_values("n_studies", ascending=False)
)

print("Pacientes con más de un estudio:", int((multi_study_df["n_studies"] > 1).sum()))
display(multi_study_df.head(20))

Pacientes con más de un estudio: 7


,patient_id,n_studies
47,100047,5
456,100456,2
455,100455,2
268,100268,2
265,100265,2
417,100417,2
416,100416,2
361,100361,1
362,100362,1
363,100363,1


## guardar manifest y split

In [31]:
manifest_path = MANIFESTS_DIR / "manifest_pancreas.csv"
split_path = MANIFESTS_DIR / "split_final_pancreas.csv"
patient_split_path = MANIFESTS_DIR / "patient_split_pancreas.csv"

manifest.to_csv(manifest_path, index=False)
split_df.to_csv(split_path, index=False)
patient_split_df.to_csv(patient_split_path, index=False)

print("Guardado:")
print(" -", manifest_path)
print(" -", split_path)
print(" -", patient_split_path)

Guardado:
 - /mnt/d/Universidad/analitica/proyecto_analitica2/data/manifests/manifest_pancreas.csv
 - /mnt/d/Universidad/analitica/proyecto_analitica2/data/manifests/split_final_pancreas.csv
 - /mnt/d/Universidad/analitica/proyecto_analitica2/data/manifests/patient_split_pancreas.csv


## resumen final

In [32]:
print("Resumen final PANCREAS manifest")
print("-" * 40)
print("Estudios totales:", len(manifest))
print("Pacientes totales:", manifest["patient_id"].nunique())
print("PDAC:", int((manifest["label_binary"] == 1).sum()))
print("non-PDAC:", int((manifest["label_binary"] == 0).sum()))
print("Con manual label:", int(manifest["manual_label_path"].notna().sum()))
print("Con automatic label:", int(manifest["automatic_label_path"].notna().sum()))

Resumen final PANCREAS manifest
----------------------------------------
Estudios totales: 557
Pacientes totales: 547
PDAC: 158
non-PDAC: 399
Con manual label: 108
Con automatic label: 449
